# 02 — Preprocessing Pipeline

Transforms the raw NIfTI scan into a clean, model-ready volume via six sequential steps:

| Step | Operation | Tool |
|------|-----------|------|
| 1 | Load as tensor with metadata | MONAI `LoadImage` |
| 2 | Reorient to RAS+ standard | MONAI `Orientation` |
| 3 | Resample to 1mm³ isotropic | MONAI `Spacing` |
| 4 | Skull stripping | deepbet (deep learning) |
| 5 | Z-score intensity normalization | numpy |
| 6 | Save preprocessed volume | nibabel |

**Output:** `data/processed/brain_preprocessed.nii.gz` — the file passed to the segmentation model in notebook 03.

In [ ]:
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from monai.transforms import (
    Compose,
    LoadImage,
    EnsureChannelFirst,
    Orientation,
    Spacing,
)
from deepbet import run_bet

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT     = Path("..").resolve()
FIGURES  = ROOT / "outputs" / "figures"
RAW      = ROOT / "data" / "raw"       / "brain_scan.nii.gz"
PROC     = ROOT / "data" / "processed"
PROC.mkdir(exist_ok=True)

RESAMPLED    = PROC / "brain_resampled.nii.gz"
STRIPPED     = PROC / "brain_stripped.nii.gz"
MASK         = PROC / "brain_stripped_mask.nii.gz"
TIV          = PROC / "brain_tiv.csv"
PREPROCESSED = PROC / "brain_preprocessed.nii.gz"

assert RAW.exists(), f"Raw scan not found: {RAW}"
print(f"Input  : {RAW}")
print(f"Size   : {RAW.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Reusable visualization helper ────────────────────────────────────────────
def plot_middle_slices(volume, title="Middle Slices", cmap="gray", save_path=None):
    """Plot the centre slice of a 3-D volume in all three anatomical planes."""
    x_mid, y_mid, z_mid = [s // 2 for s in volume.shape]

    slices = {
        "Sagittal  (axis 0)": volume[x_mid, :, :],
        "Coronal   (axis 1)": volume[:, y_mid, :],
        "Axial     (axis 2)": volume[:, :, z_mid],
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for ax, (label, sl) in zip(axes, slices.items()):
        ax.imshow(np.rot90(sl), cmap=cmap, origin="lower")
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()

## Step 1 — Load with MONAI

`LoadImage` wraps nibabel and returns a **MetaTensor**: a PyTorch tensor that carries the NIfTI affine, spacing, and header metadata alongside the voxel data. This is MONAI's core data primitive — everything downstream stays metadata-aware.

In [ ]:
load_transforms = Compose([
    LoadImage(image_only=True),   # returns MetaTensor, shape (X, Y, Z)
    EnsureChannelFirst(),          # → shape (1, X, Y, Z)  [channel-first convention]
])

img = load_transforms(str(RAW))

print(f"Shape (C, X, Y, Z) : {img.shape}")
print(f"Dtype              : {img.dtype}")
print(f"Affine             :\n{img.affine}")

## Step 2 — Reorient to RAS+

RAS+ (Right-Anterior-Superior) is the coordinate convention expected by virtually all neuroimaging DL models. The transform permutes and flips axes if needed, and updates the affine accordingly.

In [ ]:
img_ras = Orientation(axcodes="RAS")(img)

print(f"Shape after reorientation : {img_ras.shape}")
print("Orientation               : RAS+ confirmed")

## Step 3 — Resample to 1mm³ Isotropic

Pre-trained models are trained on volumes with uniform 1×1×1 mm voxels. Resampling standardises the spatial resolution regardless of the original scanner protocol. Bilinear interpolation preserves intensity continuity for MRI data.

In [ ]:
img_1mm = Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear")(img_ras)

# Verify voxel size from the updated affine
vox_size = np.sqrt((img_1mm.affine.numpy()[:3, :3] ** 2).sum(axis=0))
print(f"Shape after resampling : {img_1mm.shape}")
print(f"Voxel size             : {vox_size.round(3)} mm")

plot_middle_slices(
    img_1mm.numpy()[0],
    title="Step 3 — Resampled to 1mm³ Isotropic",
    save_path=str(FIGURES / "02_resampled.png"),
)

# Save to disk — deepbet works on NIfTI files, not tensors
nib.save(
    nib.Nifti1Image(img_1mm.numpy()[0].astype(np.float32), img_1mm.affine.numpy()),
    str(RESAMPLED),
)
print(f"\nSaved → {RESAMPLED}")

## Step 4 — Skull Stripping (deepbet)

Skull stripping removes non-brain tissue (skull, scalp, eyes) so the segmentation model only sees brain voxels. We use **deepbet**, a 2024 deep learning tool based on a lightweight CNN. In benchmarks it achieves 99.0% Dice accuracy and is ~27× faster than HD-BET.

**Outputs:**
- `brain_stripped.nii.gz` — brain tissue only  
- `brain_stripped_mask.nii.gz` — binary brain mask  
- `brain_tiv.csv` — total intracranial volume (cm³)  

*Runs on CPU — expect ~10–30 seconds on Apple Silicon.*

In [ ]:
print("Running deepbet skull stripping...")

run_bet(
    input_paths=[str(RESAMPLED)],
    brain_paths=[str(STRIPPED)],
    mask_paths= [str(MASK)],
    tiv_paths=  [str(TIV)],
    threshold=0.5,  # probability threshold for brain / non-brain voxels
    n_dilate=0,     # mask dilation in voxels (0 = tight fit)
    no_gpu=True,    # CPU mode — compatible with Apple Silicon
)

print("Done.")
print("\nTotal Intracranial Volume:")
print(pd.read_csv(TIV).to_string(index=False))

In [ ]:
stripped_data = nib.load(str(STRIPPED)).get_fdata()
mask_data     = nib.load(str(MASK)).get_fdata()
raw_data      = nib.load(str(RESAMPLED)).get_fdata()

# 3-plane view of the stripped brain
plot_middle_slices(
    stripped_data,
    title="Step 4 — Skull-Stripped Brain (deepbet)",
    save_path=str(FIGURES / "02_skull_stripped.png"),
)

# Before / After / Mask — axial slice comparison
z_mid = stripped_data.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Skull Stripping — Axial Comparison", fontsize=13, fontweight="bold")

axes[0].imshow(np.rot90(raw_data[:, :, z_mid]),     cmap="gray", origin="lower")
axes[0].set_title("Before (resampled)")
axes[1].imshow(np.rot90(stripped_data[:, :, z_mid]), cmap="gray", origin="lower")
axes[1].set_title("After Skull Stripping")
axes[2].imshow(np.rot90(mask_data[:, :, z_mid]),     cmap="hot",  origin="lower")
axes[2].set_title("Brain Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.savefig(str(FIGURES / "02_skull_strip_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## Step 5 — Intensity Normalization (Z-score)

Raw MRI intensities are arbitrary — the same scanner produces different values on different days. Z-score normalization shifts brain voxels to zero mean and unit standard deviation, making the volume scanner-agnostic and compatible with model weights trained on other scanners.

**Key detail:** statistics are computed *only on brain voxels* (inside the mask). Background remains 0.

In [ ]:
brain_mask = mask_data > 0
mean_val   = stripped_data[brain_mask].mean()
std_val    = stripped_data[brain_mask].std()

normalized = np.zeros_like(stripped_data, dtype=np.float32)
normalized[brain_mask] = (stripped_data[brain_mask] - mean_val) / std_val

print(f"Pre-normalization  → mean: {mean_val:.2f},  std: {std_val:.2f}")
print(f"Post-normalization → mean: {normalized[brain_mask].mean():.4f}, std: {normalized[brain_mask].std():.4f}")

# Histogram: before vs after
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Step 5 — Intensity Normalization", fontsize=13, fontweight="bold")

axes[0].hist(stripped_data[brain_mask].ravel(), bins=200, color="steelblue",  edgecolor="none")
axes[0].set_title("Before Normalization")
axes[0].set_xlabel("Raw Intensity")

axes[1].hist(normalized[brain_mask].ravel(),    bins=200, color="darkorange", edgecolor="none")
axes[1].set_title("After Z-score Normalization")
axes[1].set_xlabel("Intensity (σ units)")

for ax in axes:
    ax.set_ylabel("Voxel Count")

plt.tight_layout()
plt.savefig(str(FIGURES / "02_normalization_histogram.png"), dpi=150, bbox_inches="tight")
plt.show()

## Step 6 — Save Preprocessed Volume

In [ ]:
stripped_affine = nib.load(str(STRIPPED)).affine
nib.save(nib.Nifti1Image(normalized, stripped_affine), str(PREPROCESSED))

print(f"Saved → {PREPROCESSED}")
print(f"Shape : {normalized.shape}")
print(f"Dtype : {normalized.dtype}")

In [ ]:
# ── Pipeline summary ─────────────────────────────────────────────────────────
raw_hdr = nib.load(str(RAW)).header
raw_shape = raw_hdr.get_data_shape()
raw_zoom  = raw_hdr.get_zooms()[:3]

print("=" * 52)
print("  PREPROCESSING PIPELINE — SUMMARY")
print("=" * 52)
print(f"  Input shape  : {raw_shape}")
print(f"  Output shape : {normalized.shape}")
print(f"  Voxel size   : {tuple(round(v,2) for v in raw_zoom)} mm  →  (1.0, 1.0, 1.0) mm")
print(f"  Skull        : removed (deepbet, threshold=0.5)")
print(f"  Intensity    : z-score normalized (brain voxels only)")
print(f"  Output file  : {PREPROCESSED.name}")
print("=" * 52)

plot_middle_slices(
    normalized,
    title="Final Preprocessed Volume — Ready for Segmentation",
    save_path=str(FIGURES / "02_final_preprocessed.png"),
)